In [0]:
import requests
import zipfile
import io
import pandas as pd
from datetime import datetime, timezone

In [0]:
CATALOG = "gobierno_abierto"
SCHEMA_BRONZE = "a_bronze"

API_BASE = "https://gobiernoabierto.cordoba.gob.ar/api/datos-abiertos/dato"

GTFS_DATASET_ID = 3319
BARRIOS_DATASET_ID = 118
BARRIOS_VERSION_ID = 220 

In [0]:
def get_latest_version(dataset_id: int) -> dict:
    """Devuelve la versión más reciente publicada de un dataset."""
    resp = requests.get(f"{API_BASE}/{dataset_id}/version-dato", timeout=30)
    resp.raise_for_status()
    versiones = resp.json()["results"]
    if not versiones:
        raise ValueError(f"El dataset {dataset_id} no tiene versiones publicadas.")
    # Ya vienen ordenadas por fecha de creación descendente en la API
    return versiones[0]


def get_resources(dataset_id: int, version_id) -> list:
    """Devuelve todos los recursos (archivos) de una versión de un dataset."""
    resp = requests.get(f"{API_BASE}/{dataset_id}/version-dato/{version_id}/recurso", timeout=30)
    resp.raise_for_status()
    return resp.json()["results"]


def find_resource_url(dataset_id: int, version_id, title_contains: str = None) -> str:
    """
    Busca, dentro de los recursos de una versión, el que tenga el título indicado
    (búsqueda parcial, case-insensitive). Si no se pasa filtro, devuelve el primero.
    """
    recursos = get_resources(dataset_id, version_id)
    if not recursos:
        raise ValueError(f"El dataset {dataset_id} (versión {version_id}) no tiene recursos.")

    if title_contains:
        for r in recursos:
            if title_contains.lower() in r["titulo"].lower():
                return r["url"]
        raise ValueError(
            f"No se encontró un recurso con '{title_contains}' en el título. "
            f"Disponibles: {[r['titulo'] for r in recursos]}"
        )

    return recursos[0]["url"]


def audit_columns(source_dataset_id: int, source_url: str) -> dict:
    """Columnas de trazabilidad que se agregan a toda tabla Bronze."""
    return {
        "_ingested_at": datetime.now(timezone.utc),
        "_source_dataset_id": str(source_dataset_id),
        # Se guarda la URL sin el token de firma, por prolijidad y seguridad
        "_source_url": source_url.split("?")[0],
    }


In [0]:
## Ingesta GTFS (stops.txt y routes.txt)

gtfs_version = get_latest_version(GTFS_DATASET_ID)
gtfs_url = find_resource_url(GTFS_DATASET_ID, gtfs_version["id"])

print(f"Versión GTFS resuelta: {gtfs_version['titulo']}")
print(f"URL (sin firma): {gtfs_url.split('?')[0]}")

gtfs_zip_bytes = requests.get(gtfs_url, timeout=60).content
gtfs_zip = zipfile.ZipFile(io.BytesIO(gtfs_zip_bytes))

print("Archivos dentro del GTFS:")
for name in gtfs_zip.namelist():
    print(f"  - {name}")


audit = audit_columns(GTFS_DATASET_ID, gtfs_url)

# stops.txt
stops_pdf = pd.read_csv(gtfs_zip.open("stops.txt"))
for col, val in audit.items():
    stops_pdf[col] = val

stops_df = spark.createDataFrame(stops_pdf)
stops_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA_BRONZE}.gtfs_stops")

print(f"gtfs_stops: {stops_df.count()} filas escritas.")
display(stops_df.limit(5))


# routes.txt
routes_pdf = pd.read_csv(gtfs_zip.open("routes.txt"))
for col, val in audit.items():
    routes_pdf[col] = val

routes_df = spark.createDataFrame(routes_pdf)
routes_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA_BRONZE}.gtfs_routes")

print(f"gtfs_routes: {routes_df.count()} filas escritas.")
display(routes_df.limit(5))



In [0]:
## Ingesta Barrios (CSV con WKT)

barrios_url = find_resource_url(BARRIOS_DATASET_ID, BARRIOS_VERSION_ID, title_contains="WKT")
print(f"URL Barrios (sin firma): {barrios_url.split('?')[0]}")

barrios_csv_bytes = requests.get(barrios_url, timeout=60).content
barrios_pdf = pd.read_csv(io.BytesIO(barrios_csv_bytes))

print("Columnas originales del CSV:")
print(list(barrios_pdf.columns))
barrios_pdf.head()


audit_barrios = audit_columns(BARRIOS_DATASET_ID, barrios_url)
for col, val in audit_barrios.items():
    barrios_pdf[col] = val

barrios_df = spark.createDataFrame(barrios_pdf)
barrios_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA_BRONZE}.barrios_raw")

print(f"barrios_raw: {barrios_df.count()} filas escritas.")
display(barrios_df.limit(5))

In [0]:
for tabla in ["gtfs_stops", "gtfs_routes", "barrios_raw"]:
    count = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{tabla}").count()
    print(f"{CATALOG}.{SCHEMA_BRONZE}.{tabla}: {count} filas")